In [1]:
import os
import google.generativeai as genai
import sys
sys.path.append('../..')
import utils

import panel as pn  # GUI
pn.extension()

In [2]:
# Set up Gemini API key
genai.configure(api_key="")  # Replace with your actual API key

In [3]:
#!pip install panel

In [4]:
import google.generativeai as genai

models = genai.list_models()
print([m.name for m in models])


['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-4-26b-a4b-it', 'models/gemma-4-31b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3-pro-image-preview', 'models/gemini-3-pro-image', 'models/nano-banana-pro-preview', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/lyria-3-clip-preview', 'models/lyria-3-pro-preview', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-

In [5]:
def get_completion_from_messages(messages, model="gemini-flash-latest", temperature=0.7, max_tokens=500):
    genai.configure(api_key="")  # Replace with your actual API key

    client = genai.GenerativeModel(model_name=model)  # Initialize Gemini model

    # Remove "system" role and merge instructions into user's first message
    formatted_messages = []
    system_message = ""

    for message in messages:
        if message["role"] == "system":
            system_message = message["content"]  # Store system instruction
        elif "content" in message:  
            formatted_messages.append({
                "role": message["role"],  
                "parts": [{"text": message["content"]}]  # Correct format for Gemini
            })
        else:
            raise ValueError(f"Message missing 'content' field: {message}")  # Debugging

    # If there was a system message, merge it into the first user message
    if formatted_messages and system_message:
        formatted_messages[0]["parts"][0]["text"] = system_message + "\n\n" + formatted_messages[0]["parts"][0]["text"]

    response = client.generate_content(
        formatted_messages,  
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_tokens
        }
    )

    return response.text


In [6]:
# Function to process user messages
def process_user_message(user_input, all_messages, debug=True):
    import utils  # Ensure utils.py is available and correct

    delimiter = "```"

    # Step 1: Input validation
    if debug:
        print("Step 1: Input passed basic check.")

    # Step 2: Extract products and categories
    category_and_product_response = utils.find_category_and_product_only(
        user_input, utils.get_products_and_category()
    )
    print(category_and_product_response)
    category_and_product_list = utils.read_string_to_list(category_and_product_response)
    print(category_and_product_list)

    if debug:
        print(f"Step 2: Extracted products: {category_and_product_list}")

    # Step 3: Fetch product information
    product_information = utils.generate_output_string(category_and_product_list)

    if debug:
        print("Step 3: Retrieved product information.")

    # Step 4: Generate customer service response
    step_4_system_message_content = "You are a helpful AI assistant. Provide detailed and accurate information."

    messages = [
        {
            "role": "user",
            "content": step_4_system_message_content + f"\n\n{delimiter}{user_input}{delimiter}",
        },  # ✅ Merged system message
        {"role": "assistant", "content": f"Relevant product information:\n{product_information}"},
    ]

    final_response = get_completion_from_messages(all_messages + messages)
    all_messages.append({"role": "assistant", "content": final_response})

    if debug:
        print("Step 4: Generated response.")

    # Step 5: Evaluate response quality
    step_6_system_message_content = "Evaluate whether the AI response fully answers the customer's question."

    evaluation_messages = [
        {
            "role": "user",
            "content": step_6_system_message_content
            + f"\n\nCustomer message: {delimiter}{user_input}{delimiter}\n"
            + f"Agent response: {delimiter}{final_response}{delimiter}\n"
            + f"Does the response sufficiently answer the question? reply in single word Y or N",
        }
    ]

    evaluation_response = get_completion_from_messages(evaluation_messages)

    # Ensure strict formatting
    cleaned_evaluation_response = evaluation_response.strip().upper()  # Remove spaces & enforce uppercase

    if debug:
        print(f"Step 6: Evaluation result: {cleaned_evaluation_response}")

    # Step 7: Decision based on evaluation
    if cleaned_evaluation_response == "Y":
        if debug:
            print("Step 7: Response is approved.")
        return final_response, all_messages
    else:
        if debug:
            print("Step 7: Response is not sufficient.")
        return "I'm unable to provide the information you're looking for. Let me connect you with a representative.", all_messages

# Test the function
user_input = "tell me about the SmartX Pro Phone and the FotoSnap Camera, the DSLR one. Also, what tell me about your TVs?"
response, _ = process_user_message(user_input, [])
print(response)


Step 1: Input passed basic check.
[
  {
    "category": "Smartphones and Accessories",
    "products": [
      "SmartX ProPhone"
    ]
  },
  {
    "category": "Cameras and Camcorders",
    "products": [
      "FotoSnap DSLR Camera"
    ]
  },
  {
    "category": "Televisions and Home Theater Systems",
    "products": [
      "CineView 4K TV",
      "SoundMax Home Theater",
      "CineView 8K TV",
      "SoundMax Soundbar",
      "CineView OLED TV"
    ]
  }
]
[{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}, {'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'SoundMax Home Theater', 'CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV']}]
Step 2: Extracted products: [{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}, {'category': 'Televisio

InvalidArgument: 400 Requests ending with a model turn are not supported.

In [8]:
pn.extension(raw_css=['''
.assistant-response {
    background-color: #F6F6F6;
    padding: 10px;
    border-radius: 5px;
}
'''])

In [9]:
def collect_messages(debug=False):
    user_input = inp.value
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    global context

    response, context = process_user_message(user_input, context, debug=False)
    context.append({'role': 'assistant', 'content': f"{response}"})

    panels.append(pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, css_classes=['assistant-response'])))

    return pn.Column(*panels)

In [10]:
panels = []  # collect display
context = [{'role': 'system', 'content': "You are a Service Assistant"}]

inp = pn.widgets.TextInput(placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Service Assistant')
    [2] ParamFunction(function, _pane=Str, defer_load=False, height=300, loading_indicator=True)